# DermaGAN on Google Colab

Runs the full pipeline on HAM10000: lesion-level split → conditional GAN → synthetic images →
transfer learning (VGG16 / GoogLeNet / ResNet50) → Random Forest / KNN / Decision Tree → results table.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Rough timings on a free T4: GAN 100 epochs ≈ 15–25 min, each CNN run (20 epochs, frozen backbone) ≈ 10–25 min.
If you are short on time, run only ResNet50 and GoogLeNet first.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/yigitcangirgin/dermagan-ham10000.git
%cd dermagan-ham10000
!pip install -q -e .

## 1. Download HAM10000 from Kaggle

`kagglehub` downloads public datasets. If it asks for credentials, create an API token on
kaggle.com (Settings → API) and follow the prompt.

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print(DATA_DIR)

In [ ]:
!dermagan prepare --data-dir "$DATA_DIR" --out data/splits.csv

## 2. Train the conditional GAN and look at the samples (one row per class: akiec, bcc, bkl, df, mel, nv, vasc)

In [ ]:
!dermagan train-gan --epochs 100 --sample-every 10

In [ ]:
from IPython.display import Image, display
display(Image("outputs/gan/samples/epoch_100.png"))

In [ ]:
!dermagan synthesize --balance-to 1000
!for d in data/synthetic/*; do echo "$d: $(ls $d | wc -l)"; done

## 3. Transfer learning: baseline vs. + GAN images

In [ ]:
for model in ["resnet50", "googlenet", "vgg16"]:
    !dermagan train-cnn --model {model}
    !dermagan train-cnn --model {model} --synthetic data/synthetic

## 4. Classical classifiers on GoogLeNet features (as in the thesis)

In [ ]:
!dermagan extract-features --model googlenet --checkpoint outputs/cnn/googlenet/best.pt
!dermagan train-ml --features outputs/features/googlenet.npz

## 5. Results

In [ ]:
!dermagan report
from IPython.display import Markdown
display(Markdown(open("outputs/results.md").read()))
display(Image("outputs/cnn/resnet50+gan/confusion_matrix.png"))

## 6. Save everything

Downloads metrics, plots and GAN samples (not the large checkpoints). Put `results.md`, a GAN sample
grid and a confusion matrix into the README.

In [ ]:
!zip -qr results.zip outputs -x "*.pt" "*.npz"
from google.colab import files
files.download("results.zip")